### Adding volume column to convex_hull_csv

In [ ]:
"""
3D Image Analysis Pipeline - Volume and Morphological Measurements

This notebook processes 3D segmented biological data to:
1. Calculate various volume measurements (convex hull, regionprops, ellipsoid)
2. Compute morphological characteristics (Feret diameters, axis lengths)
3. Combine measurements into comprehensive output files

"""

# Library imports
import os
import math
import numpy as np
import pandas as pd
import tifffile as tiff
from skimage.measure import label, regionprops
from skimage import measure



### Measurement Functions

In [ ]:
def max_2d_feret_diameter(binary_3d_image):
    """
    Calculate the maximum 2D Feret diameter across all z-slices of a 3D segmentation.
    
    The Feret diameter is the maximum distance between any two parallel planes 
    bounding the object perpendicular to that axis.
    
    Parameters:
        binary_3d_image (np.ndarray): 3D binary array (z,y,x) where 1=foreground
        
    Returns:
        float: Maximum Feret diameter found across all z-slices
    """
    max_feret_diameter = 0
    
    for z in range(binary_3d_image.shape[0]):
        slice_2d = binary_3d_image[z, :, :]
        regions = measure.regionprops(slice_2d.astype(int))
        
        if regions:
            current_feret = regions[0].feret_diameter_max
            max_feret_diameter = max(max_feret_diameter, current_feret)
    
    return max_feret_diameter

In [ ]:
def mean_2d_feret_diameter(binary_3d_image):
    """
    Calculate the mean 2D Feret diameter across all z-slices.
    
    Parameters:
        binary_3d_image (np.ndarray): 3D binary array (z,y,x)
        
    Returns:
        float: Mean Feret diameter across slices (0 if no regions found)
    """
    feret_diameters = []
    
    for z in range(binary_3d_image.shape[0]):
        slice_2d = binary_3d_image[z, :, :]
        regions = measure.regionprops(slice_2d.astype(int))
        
        if regions:
            feret_diameters.append(regions[0].feret_diameter_max)
    
    return np.mean(feret_diameters) if feret_diameters else 0

In [ ]:
def median_2d_feret_diameter(binary_3d_image):
    """
    Calculate the median 2D Feret diameter across all z-slices.
    
    Parameters:
        binary_3d_image (np.ndarray): 3D binary array (z,y,x)
        
    Returns:
        float: Median Feret diameter across slices (0 if no regions found)
    """
    feret_diameters = []
    
    for z in range(binary_3d_image.shape[0]):
        slice_2d = binary_3d_image[z, :, :]
        regions = measure.regionprops(slice_2d.astype(int))
        
        if regions:
            feret_diameters.append(regions[0].feret_diameter_max)
    
    return np.median(feret_diameters) if feret_diameters else 0

In [ ]:
def ellipsoid_volume_from_moments(inertia_tensor):
    """
    Calculate ellipsoid volume approximation from inertia tensor moments.
    
    Parameters:
        inertia_tensor (np.ndarray): 3x3 inertia tensor matrix
        
    Returns:
        float: Volume of approximated ellipsoid
    """
    eigenvalues = np.linalg.eigvals(inertia_tensor)
    semi_axes = np.sqrt(5 * eigenvalues)  # Conversion factor from moments to lengths
    a, b, c = np.sort(semi_axes)[::-1]  # Sort axes descending
    return (4/3) * np.pi * a * b * c

In [ ]:
def measure_components(binary_mask):
    """
    Calculate comprehensive morphological measurements for all components in a 3D mask.
    
    Parameters:
        binary_mask (np.ndarray): 3D binary array where 1=foreground
        
    Returns:
        pd.DataFrame: DataFrame containing measurements for each component
    """
    labeled_array, num_features = label(binary_mask, return_num=True)
    props = regionprops(labeled_array)
    
    print(f"\nProcessing {len(props)} components...")
    
    data = {
        'volume - regionprops': [],
        'max_diameter': [],
        'max_2d_feret_diameter': [],
        'median_2d_feret_diameter': [],
        'mean_2d_feret_diameter': [],
        'sphere_volume_equivalent': [],
        'axis_major_length': [],
        'axis_minor_length': [],
        'ellipsoid_volume': []
    }
    
    for prop in props:
        print(".", end="", flush=True)
        
        volume = prop.area
        max_diameter = prop.equivalent_diameter
        axis_major_length = prop.axis_major_length
        axis_minor_length = prop.axis_minor_length
        
        max_feret = max_2d_feret_diameter(prop.image)
        median_feret = median_2d_feret_diameter(prop.image)
        mean_feret = mean_2d_feret_diameter(prop.image)
        
        ellipsoid_vol = ellipsoid_volume_from_moments(prop.inertia_tensor)
        sphere_vol = math.pi * (4/3) * ((median_feret/2)**3)
        
        data['volume - regionprops'].append(volume)
        data['max_diameter'].append(max_diameter)
        data['max_2d_feret_diameter'].append(max_feret)
        data['median_2d_feret_diameter'].append(median_feret)
        data['mean_2d_feret_diameter'].append(mean_feret)
        data['sphere_volume_equivalent'].append(sphere_vol)
        data['axis_major_length'].append(axis_major_length)
        data['axis_minor_length'].append(axis_minor_length)
        data['ellipsoid_volume'].append(ellipsoid_vol)
    
    return pd.DataFrame(data)

### File Processing Functions

In [ ]:
def add_volume_to_csv(directory):
    """
    Process a single directory to add volume measurements to existing CSV data.
    
    Parameters:
        directory (str): Path to directory containing:
            - {foldername}_volumes.csv
            - {foldername}_filtered.tiff
    """
    folder_name = os.path.basename(os.path.normpath(directory))
    
    # Define input/output paths
    input_csv = os.path.join(directory, f'{folder_name}_volumes.csv')
    output_csv = os.path.join(directory, f'{folder_name}_measurements.csv')
    binary_mask_path = os.path.join(directory, f'{folder_name}_filtered.tiff')

    # Load data
    binary_mask = tiff.imread(binary_mask_path)
    df = pd.read_csv(input_csv)
    
    # Rename existing volume column
    df.rename(columns={'Volume': 'volume - convex hull'}, inplace=True)
    
    # Calculate new measurements
    new_measurements = measure_components(binary_mask)
    
    # Merge measurements
    for col in new_measurements.columns:
        df[col] = new_measurements[col]
    
    # Save results
    df.to_csv(output_csv, index=False)
    print(f"\nSaved measurements to {output_csv}")

In [ ]:
def iterate_subfolders(root_path):
    """
    Process all valid subdirectories within a root directory.
    Skips directories containing "plots" in their name.
    
    Parameters:
        root_path (str): Root directory to search for data folders
    """
    for root, dirs, files in os.walk(root_path):
        for dir_name in dirs:
            if "plots" in dir_name:
                continue
            subfolder_path = os.path.join(root, dir_name)
            try:
                print(f"\nProcessing {subfolder_path}...")
                add_volume_to_csv(subfolder_path)
            except Exception as e:
                print(f"Error processing {subfolder_path}: {str(e)}")

In [ ]:
# Execution


if __name__ == "__main__":
    individual_paths = [
        '/sv_measurements/whopper2/measurements/whopper239/',
        '/sv_measurements/20240324/CG58slot1_SR/measurements_convex_hull_3/pos003/'
    ]
    
    # Process a single directory
    add_volume_to_csv(sample_paths[0])
    
    # Process entire directory tree
    root_directory = "/sv_measurements/20240304/CG49slot1_SR/h5_seg/measurements_convex_hull_1/"
    iterate_subfolders(root_directory)

In [1]:
import pandas as pd
from skimage.measure import label, regionprops
from skimage import measure
import tifffile as tiff
import os
import numpy as np
import math

In [2]:
def max_2d_feret_diameter(binary_3d_image):
    """
    Calculate the maximum 2D Feret diameter of a 3D binary segmentation across all z-slices.
    
    Parameters:
    binary_3d_image (numpy.ndarray): 3D binary image array
    
    Returns:
    float: Maximum 2D Feret diameter found in any z-slice
    """
    max_feret_diameter = 0
    
    
    for z in range(binary_3d_image.shape[0]):
        
        slice_2d = binary_3d_image[z, :, :]        
        regions = measure.regionprops(slice_2d.astype(int))
        
        
        if regions:
            slice_feret_diameter = regions[0].feret_diameter_max
            max_feret_diameter = max(max_feret_diameter, slice_feret_diameter)
    
    return max_feret_diameter
    
def mean_2d_feret_diameter(binary_3d_image):
    """
    Calculate the mean 2D Feret diameter of a 3D binary segmentation across all z-slices.
    
    Parameters:
    binary_3d_image (numpy.ndarray): 3D binary image array
    
    Returns:
    float: Mean 2D Feret diameter across all z-slices
    """
    feret_diameters = []
    
    for z in range(binary_3d_image.shape[0]):
        slice_2d = binary_3d_image[z, :, :]
        
        regions = measure.regionprops(slice_2d.astype(int))
        
        if regions:
            feret_diameters.append(regions[0].feret_diameter_max)
    
    return np.mean(feret_diameters) if feret_diameters else 0

def median_2d_feret_diameter(binary_3d_image):
    """
    Calculate the median 2D Feret diameter of a 3D binary segmentation across all z-slices.
    
    Parameters:
    binary_3d_image (numpy.ndarray): 3D binary image array
    
    Returns:
    float: Mean 2D Feret diameter across all z-slices
    """
    feret_diameters = []
    
    
    for z in range(binary_3d_image.shape[0]):
        slice_2d = binary_3d_image[z, :, :]
        regions = measure.regionprops(slice_2d.astype(int))
    
        if regions:
            feret_diameters.append(regions[0].feret_diameter_max)
    
    return np.median(feret_diameters) if feret_diameters else 0


def ellipsoid_volume_from_moments(inertia_tensor):
    """
    Calculate the volume of an ellipsoid approximation of a region using its moments.
    """
    
    eigenvalues = np.linalg.eigvals(inertia_tensor)
    
    # factor 5 comes from the relation between second moments and semi-axes lengths
    semi_axes = np.sqrt(5 * eigenvalues)
    
    a, b, c = np.sort(semi_axes)[::-1]
    
    volume = (4/3) * np.pi * a * b * c
    
    return volume
    
def measure_components(binary_mask):
    labeled_array, num_features = label(binary_mask, return_num=True)
    props = regionprops(labeled_array)
   
    data = {
        'volume - regionprops': [],
        'max_diameter': [],
        #'max_area': [],
        'max_2d_feret_diameter' : [],
        'median_2d_feret_diameter' : [],
        'mean_2d_feret_diameter' : [],
        'sphere_volume_equivalent' : [],
        'axis_major_length': [],
        'axis_minor_length': [],
        'ellipsoid_volume': []
    }
    print("\nNumber of components:", len(props))
    for prop in props:
        print(".", end="")
        volume = prop.area
        max_diameter = prop.equivalent_diameter
        feret_diameter = prop.feret_diameter_max
        axis_major_length = prop.axis_major_length
        axis_minor_length = prop.axis_minor_length
        
        max_2d_feret_diam = max_2d_feret_diameter(prop.image)
        median_2d_feret_diam = median_2d_feret_diameter(prop.image)
        mean_2d_feret_diam = mean_2d_feret_diameter(prop.image)
        ellipsoid_volume = ellipsoid_volume_from_moments(prop.inertia_tensor)
        
        data['volume - regionprops'].append(volume)
        data['max_diameter'].append(max_diameter)
        data['max_2d_feret_diameter'].append(max_2d_feret_diam)
        data['median_2d_feret_diameter'].append(median_2d_feret_diam)
        data['mean_2d_feret_diameter'].append(mean_2d_feret_diam)
        data['sphere_volume_equivalent'].append(math.pi * (4/3) * ((median_2d_feret_diam/2)**3))
        data['axis_major_length'].append(prop.axis_major_length)
        data['axis_minor_length'].append(axis_minor_length)
        data['ellipsoid_volume'].append(ellipsoid_volume)
    return pd.DataFrame(data)

def add_volume_to_csv(directory):

    folder_name = os.path.basename(os.path.normpath(directory))
    
    input_csv = os.path.join(directory, f'{folder_name}_volumes.csv')
    output_csv = os.path.join(directory, f'{folder_name}_measurements.csv')
    binary_mask_path = os.path.join(directory, f'{folder_name}_filtered.tiff')

    binary_mask = tiff.imread(binary_mask_path)
    df = pd.read_csv(input_csv)
    
    df.rename(columns={'Volume': 'volume - convex hull '}, inplace=True) # Rename current "volume" column
    
    new_volumes_df = measure_components(binary_mask)
    

    df['volume - regionprops'] = new_volumes_df['volume - regionprops']
    df['max_diameter'] = new_volumes_df['max_diameter']
    #df['max_area'] = new_volumes_df['max_area']
    df['max_2d_feret_diameter'] = new_volumes_df['max_2d_feret_diameter']
    df['median_2d_feret_diameter'] = new_volumes_df['median_2d_feret_diameter']
    df['mean_2d_feret_diameter'] = new_volumes_df['mean_2d_feret_diameter']
    df['sphere_volume_equivalent'] = new_volumes_df['sphere_volume_equivalent']
    df['axis_major_length'] = new_volumes_df['axis_major_length']
    df['axis_minor_length'] = new_volumes_df['axis_minor_length']
    df['ellipsoid_volume'] = new_volumes_df['ellipsoid_volume']
    df.to_csv(output_csv, index=False)


In [7]:

path_files = '/sv_measurements/20240324/CG52slot2_SR/measurements2/pos003/'

add_volume_to_csv(path_files)


Number of components: 73
.........................................................................

In [8]:

def iterate_subfolders(root_path):
    for root, dirs, files in os.walk(root_path):
        for dir_name in dirs:
            if "plots" in dir_name:
                continue
            subfolder_path = os.path.join(root, dir_name)
            add_volume_to_csv(subfolder_path)
            
root_directory = "/sv_measurements/20240324/CG52slot2_SR/measurements2/"  

iterate_subfolders(root_directory)


Number of components: 13
.............
Number of components: 41
.........................................
Number of components: 75
...........................................................................
Number of components: 73
.........................................................................
Number of components: 70
......................................................................
Number of components: 38
......................................